# 04 · Gold: star schema

Reshapes the silver tables into a star schema for Power BI: one narrow fact table, one bridge table and five dimensions.

- **Input:** `lh_silver.dbo.ads`, `lh_silver.dbo.ad_skills`
- **Output (`lh_gold.dbo`):** `fact_ad`, `fact_ad_skill`, `dim_date`, `dim_role`, `dim_location`, `dim_employer`, `dim_skill`
- **How to run:** *Run all*. Every table is overwritten, so re-running gives the same result.
- **Lakehouses:** `lh_gold` attached as default, `lh_silver` attached as well.

```
                dim_date   dim_role   dim_location   dim_employer
                    \          |           |            /
                     +-------- fact_ad (one row per ad) -+
                                   |
                            fact_ad_skill (bridge: ad × skill)
                                   |
                               dim_skill
```

## Design notes

- **Why a star schema.** Silver is wide and contains free text. Power BI works best with a narrow fact table of keys and numbers, and small dimension tables holding the descriptive attributes once each. Filters on a dimension then propagate to the facts automatically.
- **Skills are many-to-many with ads**, so they live in a bridge table (`fact_ad_skill`, one row per ad and skill) rather than as columns on `fact_ad`.
- **A full date dimension** (every day 2016–2026), not just the dates that occur. Years or months without ads for a role then show as zero rather than disappearing — for example Analytics Engineer has no ads in 2018.
- **Unknown members instead of NULL.** Missing region, municipality or employer become `Unknown`, so those ads still join to their dimension and stay visible when a report is filtered on location.
- **Region repaired from municipality.** 27 municipalities (including Stockholm, Göteborg and Malmö) appeared with no region in some ads. The region is inferred from the other ads for the same municipality, using the most common region per municipality. Afterwards only one `Unknown`/`Unknown` location remains, for ads with neither field.
- **Surrogate keys** come from `row_number()` over a sorted list of distinct values. That pulls each dimension onto one Spark node, which is fine at this size (at most ~2,000 rows); a very large dimension would use a hash or `monotonically_increasing_id()` instead.
- **The ad text stays in silver.** It is not needed for the analysis and would make the model heavy. Headline and URL are kept on `fact_ad` so that a report can list and link to individual ads.
- **Verification** checks referential integrity: no ads lost, no missing keys, no orphan bridge rows, no dates outside the date dimension.


## 1. Setup and region repair

In [1]:
from pyspark.sql import functions as F, Window as W

ads_raw = spark.table("lh_silver.dbo.ads")
ad_skills = spark.table("lh_silver.dbo.ad_skills")
GOLD = "lh_gold.dbo"
DATE_START, DATE_END = "2016-01-01", "2026-12-31"


def write(df, name):
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{GOLD}.{name}"))
    print(f"{name:16} {spark.table(f'{GOLD}.{name}').count():>7} rows")


# Some ads have a municipality but no region, although the municipality determines
# the region. Infer it from the ads that have both (most common region per municipality).
municipality_region = (ads_raw.filter("region is not null and municipality is not null")
    .groupBy("municipality", "region").count()
    .withColumn("rank", F.row_number().over(
        W.partitionBy("municipality").orderBy(F.desc("count"))))
    .filter("rank = 1")
    .select("municipality", F.col("region").alias("region_lookup")))

ads = (ads_raw
    .join(F.broadcast(municipality_region), "municipality", "left")
    .withColumn("region", F.coalesce("region", "region_lookup", F.lit("Unknown")))
    .withColumn("municipality", F.coalesce("municipality", F.lit("Unknown")))
    .withColumn("employer_name", F.coalesce("employer_name", F.lit("Unknown")))
    .withColumn("employer_orgnr", F.coalesce("employer_orgnr", F.lit("Unknown")))
    .drop("region_lookup"))

print("Ads without region before repair:", ads_raw.filter("region is null").count())
print("Ads with region Unknown after repair:", ads.filter("region = 'Unknown'").count())

StatementMeta(, 1e488d33-9230-43b3-9ce9-cd3e0377c153, 3, Finished, Available, Finished, False)

Ads without region before repair: 1714
Ads with region Unknown after repair: 1485


## 2. Date dimension

In [2]:
dim_date = (spark.sql(f"""
        SELECT explode(sequence(to_date('{DATE_START}'), to_date('{DATE_END}'), interval 1 day)) AS date
    """)
    .select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("date"),
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.date_format("date", "MMM").alias("month_name"),
    ))
write(dim_date, "dim_date")

StatementMeta(, 1e488d33-9230-43b3-9ce9-cd3e0377c153, 4, Finished, Available, Finished, False)

dim_date            4018 rows


## 3. Other dimensions

In [3]:
def make_dim(df, cols, key):
    """Distinct combinations of cols, with a sequential surrogate key."""
    distinct = df.select(*cols).distinct()
    return distinct.withColumn(key, F.row_number().over(W.orderBy(*cols)))

dim_role     = make_dim(ads, ["role"], "role_key")
dim_location = make_dim(ads, ["region", "municipality"], "location_key")
dim_employer = make_dim(ads, ["employer_name", "employer_orgnr"], "employer_key")
dim_skill    = make_dim(ad_skills, ["skill", "category"], "skill_key")

for d, n in [(dim_role, "dim_role"), (dim_location, "dim_location"),
             (dim_employer, "dim_employer"), (dim_skill, "dim_skill")]:
    write(d, n)

StatementMeta(, 1e488d33-9230-43b3-9ce9-cd3e0377c153, 5, Finished, Available, Finished, False)

dim_role               5 rows
dim_location         131 rows
dim_employer        2038 rows
dim_skill             46 rows


## 4. Fact table

In [4]:
fact_ad = (ads
    .join(F.broadcast(dim_role), "role")
    .join(F.broadcast(dim_location), ["region", "municipality"])
    .join(dim_employer, ["employer_name", "employer_orgnr"])
    .select(
        "ad_id",
        F.date_format("published_at", "yyyyMMdd").cast("int").alias("date_key"),
        "role_key", "location_key", "employer_key",
        F.coalesce("n_vacancies", F.lit(1)).alias("n_vacancies"),
        F.datediff("application_deadline", "published_at").alias("days_open"),
        "headline", "webpage_url",
    ))
write(fact_ad, "fact_ad")

StatementMeta(, 1e488d33-9230-43b3-9ce9-cd3e0377c153, 6, Finished, Available, Finished, False)

fact_ad            11718 rows


## 5. Bridge table

In [5]:
fact_ad_skill = (ad_skills
    .join(F.broadcast(dim_skill), ["skill", "category"])
    .select("ad_id", "skill_key"))
write(fact_ad_skill, "fact_ad_skill")

StatementMeta(, 1e488d33-9230-43b3-9ce9-cd3e0377c153, 7, Finished, Available, Finished, False)

fact_ad_skill      46015 rows


## 6. Verify referential integrity

In [6]:
f = spark.table(f"{GOLD}.fact_ad")
b = spark.table(f"{GOLD}.fact_ad_skill")
d = spark.table(f"{GOLD}.dim_date")

n_silver = ads_raw.count()
n_fact = f.count()
n_missing_keys = f.filter("date_key is null or role_key is null or location_key is null or employer_key is null").count()
n_orphans = b.join(f, "ad_id", "left_anti").count()
n_outside_dates = f.join(d, "date_key", "left_anti").count()

print(f"Ads in silver: {n_silver}, in fact_ad: {n_fact}")
print(f"Missing keys: {n_missing_keys}")
print(f"Bridge rows without a matching ad: {n_orphans}")
print(f"Dates outside dim_date: {n_outside_dates}")

assert n_fact == n_silver, "Ads were lost in the fact table"
assert n_missing_keys == 0, "Fact rows with missing keys"
assert n_orphans == 0, "Orphan rows in the bridge table"
assert n_outside_dates == 0, "Publication dates outside dim_date — extend DATE_END"

StatementMeta(, 1e488d33-9230-43b3-9ce9-cd3e0377c153, 8, Finished, Available, Finished, False)

Ads in silver: 11718, in fact_ad: 11718
Missing keys: 0
Bridge rows without a matching ad: 0
Dates outside dim_date: 0
